# Exercise 4 — play with the 6 × 6 grid

Same net as Exercise 3. The four corners are fixed, all other nodes are free. Change one thing at a time:

- the boundary conditions in the z-axis
- the force densities `q`
- the external loads `p`

## 1. The net

In [1]:
import numpy as np
from session_py import Color, Matrix, Point, Session, Xform
from session_jupyter import show

# nodes: 6 x 6 cells = 7 x 7 nodes, numbered row by row from the bottom left
n = 7
x, y = np.meshgrid(range(n), range(n))
x = x.flatten().astype(float)
y = y.flatten().astype(float)
z = np.zeros(n * n)

# bars: each node to its right and upper neighbour
edges = [(k, k+1) for k in range(n*n) if x[k] < n-1] + \
        [(k, k+n) for k in range(n*n) if y[k] < n-1]

# connectivity matrix
C = np.zeros((len(edges), n*n))
for e, (a, b) in enumerate(edges):
    C[e, a], C[e, b] = 1, -1

# fixed = 4 corners, free = everything else
fixed = []
free  = []
for k in range(n * n):
    if x[k] in (0, n-1) and y[k] in (0, n-1):
        fixed.append(k)
    else:
        free.append(k)

C_N, C_F = C[:, free], C[:, fixed]

## 2. Solve and draw

The same two steps are needed after every change, so they go into two functions.

In [2]:
def form_find(z, q, p_z):
    """Move the free nodes to equilibrium. Returns the new x, y, z and the bar forces f."""
    Q = np.diag(q)
    x_new, y_new, z_new = x.copy(), y.copy(), z.copy()

    # equation 6.32
    A = C_N.T @ Q @ C_N
    x_new[free] = np.linalg.solve(A, 0   - C_N.T @ Q @ C_F @ x[fixed])
    y_new[free] = np.linalg.solve(A, 0   - C_N.T @ Q @ C_F @ y[fixed])
    z_new[free] = np.linalg.solve(A, p_z - C_N.T @ Q @ C_F @ z[fixed])

    # bar forces f = q l
    f = q * np.sqrt((C @ x_new)**2 + (C @ y_new)**2 + (C @ z_new)**2)
    return x_new, y_new, z_new, f


def draw(x, y, z, f, name):
    """Show the net in the viewer. Bar colour = bar force (blue = small, red = big)."""
    colors = []
    for force in f:
        t = force / f.max()
        colors.append(Color(t, 0.0, 1.0 - t, 1.0))

    session = Session(name)
    session.add_network(Matrix.from_rows(C.tolist()), Matrix.from_rows(np.column_stack([x, y, z]).tolist()), colors, [3.0] * len(edges))

    # the four fixed corners, bigger
    for k in fixed:
        point = Point(x[k], y[k], z[k])
        point.width = 14.0
        session.add_point(point)

    return show(session, name)

## 3. Start: flat corners, q = 1, no load

The net stays flat. All bars pull equally hard per length, so the edges are sucked inwards.

In [3]:
q   = np.ones(len(edges))
p_z = np.zeros(len(free))

x1, y1, z1, f1 = form_find(z, q, p_z)
draw(x1, y1, z1, f1, "fdm_exercise_4_start")

## 4. Change the boundary conditions in z

Lift two opposite corners by 4. The flat net becomes a saddle.

In [4]:
z[0]       = 4      # corner (0, 0)
z[n*n - 1] = 4      # corner (6, 6)

x2, y2, z2, f2 = form_find(z, q, p_z)
draw(x2, y2, z2, f2, "fdm_exercise_4_corners")

## 5. Change the force densities q

Give the bars on the four edges `q = 10`. The edge cables pull harder, get straighter and stretch the net open.

In [5]:
for e, (a, b) in enumerate(edges):
    if (x[a] == x[b] and x[a] in (0, n-1)) or (y[a] == y[b] and y[a] in (0, n-1)):
        q[e] = 10       # edge cables pull harder

x3, y3, z3, f3 = form_find(z, q, p_z)
draw(x3, y3, z3, f3, "fdm_exercise_4_q")

## 6. Change the external loads p

Hang a load on every free node. The net sags; x and y do not change, because the load is vertical.

In [6]:
p_z = -0.3 * np.ones(len(free))

x4, y4, z4, f4 = form_find(z, q, p_z)
draw(x4, y4, z4, f4, "fdm_exercise_4_load")

In [7]:
middle = (n * n) // 2     # node in the middle of the net
print("height of the middle node before the load:", round(z3[middle], 2))
print("height of the middle node with the load:  ", round(z4[middle], 2))
print("biggest bar force before the load:", round(f3.max(), 2))
print("biggest bar force with the load:  ", round(f4.max(), 2))

height of the middle node before the load: 2.0
height of the middle node with the load:   0.96
biggest bar force before the load: 12.48
biggest bar force with the load:   13.5


Every viewer shows a net in exact equilibrium, found with one linear solve. Now change the numbers yourself: other corners, other `q`, a load in the other direction (`p_z = +0.3` gives a dome-like net).